# What is SQL?

SQL (Structured Query Language) is used to:

- Store data
- Retrieve data
- Update data
- Delete data

In simple terms: **SQL helps us manage data in databases**

---

# What is CRUD?

CRUD stands for:

| Operation | Meaning | SQL Command |
| --- | --- | --- |
| C | Create | INSERT |
| R | Read | SELECT |
| U | Update | UPDATE |
| D | Delete | DELETE |

---


## How SQL works

SQL is **declarative**: we describe *what data we want* or *what change we want*, and the database engine decides how to execute it.

For this class, think of the four CRUD operations like this:

| SQL | Purpose | PySpark comparison |
| --- | --- | --- |
| `INSERT` | Add rows | `union()` / `unionByName()` |
| `SELECT` | Read columns/rows | `select()` |
| `UPDATE` | Change column values | `withColumn()` |
| `DELETE` | Remove rows | `filter()` |

The important difference is that SQL works with **tables**, while PySpark commonly works with **DataFrames**.

# 1. CREATE → INSERT Data

### Purpose:

Add new data into a table

### Syntax:

```
INSERT INTO table_name (column1, column2, column3)
VALUES (value1, value2, value3);
```

### Schema and temporary table

A **schema** is a logical container for database objects such as tables and views.

A **temporary table** is session-scoped. It is useful for practice and intermediate work because it is not intended to become a permanent table in the catalog.

For comparison:

- SQL: `CREATE TEMPORARY TABLE customers (...)`
- PySpark: a DataFrame is an in-memory/distributed dataset, and `createOrReplaceTempView()` exposes it to SQL as a temporary view.

In [0]:
CREATE SCHEMA IF NOT EXISTS b2practice.sql

In [0]:
CREATE OR REPLACE TEMPORARY TABLE customers (
    customer_id INT,
    name STRING,
    country STRING,
    signup_date DATE
);

In [0]:
INSERT INTO customers
VALUES
    (1, 'John Smith', 'UK', '2024-01-10');

num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO customers
VALUES
    (2, 'Alice Brown', 'USA', '2024-02-15'),
    (3, 'Raj Patel', 'India', '2024-03-01'),
    (4, 'Emma Wilson', 'UK', '2024-04-01');

num_affected_rows,num_inserted_rows
3,3


In [0]:
select * from customers

customer_id,name,country,signup_date
2,Alice Brown,USA,2024-02-15
3,Raj Patel,India,2024-03-01
4,Emma Wilson,UK,2024-04-01
1,John Smith,UK,2024-01-10


## Key Points:

- Column names must match values
- Data types should be correct
- You can insert multiple rows

## Practice:

- Insert 5 customers from different countries

### Understanding SELECT

`SELECT` is the main SQL command for reading data.

A query normally has two important parts:

```sql
SELECT columns
FROM table;
```

You can think of it as:

- `SELECT` → **which columns do I want?**
- `FROM` → **which table do I want them from?**

This maps closely to PySpark:

```python
df.select("name", "country")
```

# 2. READ → SELECT Data

### Purpose:

Retrieve data from a table

### Syntax:

```
SELECT column1, column2
FROM table_name;
```

In [0]:
SELECT name, country FROM customers;

name,country
Alice Brown,USA
Raj Patel,India
Emma Wilson,UK
John Smith,UK


In [0]:
-- Filtering Data (WHERE)

SELECT * FROM customers
WHERE country='UK';

customer_id,name,country,signup_date
4,Emma Wilson,UK,2024-04-01
1,John Smith,UK,2024-01-10


### Key Points:

- `*` means all columns
- WHERE filters data
- Conditions matter

### Practice:

- Show all customers
- Show only UK customers
- Show only names

---

### Understanding UPDATE

`UPDATE` changes values in existing rows.

The `SET` clause defines **what changes**, while `WHERE` defines **which rows change**.

This is different from the usual PySpark DataFrame pattern because DataFrames are immutable. In PySpark, we normally create a new column expression and assign the resulting DataFrame back to `df`.

```text
SQL                         PySpark
UPDATE ... SET ... WHERE    withColumn(..., when(...))
```

# 3. UPDATE → Modify Data

### Purpose:

Change existing data

### 🧾 Syntax:


```
UPDATE table_name
SET column=value
WHERE condition;
```


In [0]:
UPDATE customers
SET country='USA'
WHERE customer_id=1;

### Important Warning:

```
UPDATE customers SET country='India';
```

👉 This will update **ALL rows**

### Key Points:

- Always use WHERE
- Without WHERE = dangerous

### Practice:

- Update a customer’s country
- Update multiple customers

### Understanding DELETE

`DELETE` removes rows from a table.

The `WHERE` clause is critical because it controls which rows are removed.

In PySpark, the common equivalent is to **keep the rows you want** using `filter()`:

```python
df = df.filter(df.customer_id != 1)
```

So the mental model is:

- SQL: `DELETE ... WHERE condition`
- PySpark: `filter(NOT condition)` / filter the rows you want to keep

# 4. DELETE → Remove Data

### Purpose:

Delete data from table

## 🧾 Syntax:

```
DELETE FROM table_name
WHERE condition;
```

## Example:

In [0]:
Delete FROM customers
WHERE country='UK';

num_affected_rows
2


In [0]:
select * from customers

customer_id,name,country,signup_date
2,Alice Brown,USA,2024-02-15
3,Raj Patel,India,2024-03-01


## Adding and changing columns

There are two common SQL patterns:

1. **Calculate a column in a query** without changing the table.
2. **Actually add a column to the table** with `ALTER TABLE`, then populate it with `UPDATE`.

For example, `YEAR(signup_date)` calculates the year from a date.

PySpark comparison:

```python
from pyspark.sql.functions import year

df = df.withColumn("signup_year", year("signup_date"))
```

`ALTER TABLE` changes the table schema, whereas `SELECT ... AS` only changes the result of that query.

# Adding columns

In [0]:
SELECT
    *,
    YEAR(signup_date) AS signup_year
FROM customers;

customer_id,name,country,signup_date,signup_year
2,Alice Brown,USA,2024-02-15,2024
3,Raj Patel,India,2024-03-01,2024


In [0]:
ALTER TABLE customers
ADD COLUMNS (signup_year INT);

UPDATE customers
SET signup_year = YEAR(signup_date);

In [0]:
ALTER TABLE customers
RENAME COLUMN name TO customer_name;

# 5. DISTINCT → Remove Duplicate Results

### Purpose

`DISTINCT` returns only unique combinations of the selected columns.

### SQL

```sql
SELECT DISTINCT country
FROM customers;
```

If you select more than one column, uniqueness is checked across the combination of those columns.

### PySpark comparison

```python
df.select("country").distinct()
```

## Practice

- Show the distinct countries.
- Show distinct combinations of `country` and `signup_date`.


In [ ]:
SELECT DISTINCT country
FROM customers;

# 6. ORDER BY → Sort Data

### Purpose

`ORDER BY` sorts the result.

- `ASC` = ascending (default)
- `DESC` = descending

### SQL

```sql
SELECT *
FROM customers
ORDER BY signup_date DESC;
```

### PySpark comparison

```python
df.orderBy("signup_date")
df.orderBy(df.signup_date.desc())
```

## Practice

- Sort customers by `signup_date` from newest to oldest.
- Sort by `country` ascending and `customer_id` descending.


In [ ]:
SELECT *
FROM customers
ORDER BY signup_date DESC;

# 7. WHERE → Multiple Conditions

### Purpose

`WHERE` can combine conditions using `AND`, `OR`, and `NOT`.

### SQL

```sql
SELECT *
FROM customers
WHERE country = 'UK'
   OR country = 'USA';
```

### PySpark comparison

```python
df.filter(
    (df.country == "UK") | (df.country == "USA")
)
```

Remember that PySpark uses `&` and `|` for DataFrame boolean expressions.

## Practice

- Find customers from the UK or USA.
- Find customers whose `customer_id` is greater than 2 and country is India.


In [ ]:
SELECT *
FROM customers
WHERE country IN ('UK', 'USA');

# 8. NULL Handling

### Purpose

SQL uses `NULL` to represent a missing/unknown value.

Use `IS NULL` and `IS NOT NULL` rather than `= NULL`.

### SQL

```sql
SELECT *
FROM customers
WHERE country IS NULL;
```

### PySpark comparison

```python
df.filter(df.country.isNull())
df.filter(df.country.isNotNull())
```

For replacing missing values in PySpark, `fillna()` is commonly used.

## Practice

- Find customers where `country` is missing.
- Find customers where `country` is not missing.


In [ ]:
SELECT *
FROM customers
WHERE country IS NOT NULL;

# 9. GROUP BY → Aggregate Data

### Purpose

`GROUP BY` groups rows so that aggregate functions can calculate values per group.

Common aggregate functions:

```text
COUNT()
SUM()
AVG()
MIN()
MAX()
```

### SQL

```sql
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country;
```

### PySpark comparison

```python
from pyspark.sql.functions import count

df.groupBy("country").agg(
    count("*").alias("customer_count")
)
```

## Practice

- Count customers in each country.
- Find the earliest signup date for each country.


In [ ]:
SELECT
    country,
    COUNT(*) AS customer_count
FROM customers
GROUP BY country;

# 10. HAVING → Filter Groups

### Purpose

`WHERE` filters **rows before grouping**.

`HAVING` filters **groups after aggregation**.

### SQL

```sql
SELECT country, COUNT(*) AS customer_count
FROM customers
GROUP BY country
HAVING COUNT(*) > 1;
```

### PySpark comparison

PySpark commonly uses `filter()` after the aggregation:

```python
df.groupBy("country")   .count()   .filter("count > 1")
```

## Practice

- Return only countries having more than one customer.


In [ ]:
SELECT
    country,
    COUNT(*) AS customer_count
FROM customers
GROUP BY country
HAVING COUNT(*) > 1;

# 11. JOIN → Combine Tables

### Purpose

A `JOIN` combines rows from two tables using a related column.

The most common types are:

| Join | Meaning |
| --- | --- |
| `INNER JOIN` | Matching rows from both tables |
| `LEFT JOIN` | All rows from left + matches from right |
| `RIGHT JOIN` | All rows from right + matches from left |
| `FULL OUTER JOIN` | All rows from both sides |

### SQL

```sql
SELECT c.customer_id, c.name, o.order_id
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id;
```

### PySpark comparison

```python
customers.join(
    orders,
    "customer_id",
    "left"
)
```

## Practice

Assume an `orders` DataFrame/table exists.

- Perform an inner join using `customer_id`.
- Perform a left join and keep all customers.


In [ ]:
-- Example: assumes an orders table exists
SELECT
    c.customer_id,
    c.name,
    o.order_id
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id;

# 12. UNION → Combine Rows

### Purpose

`UNION` combines the result of two queries **vertically**.

The queries must have compatible column structures.

`UNION` removes duplicate rows.

`UNION ALL` keeps duplicates and is commonly used when you want every row from both inputs.

### PySpark comparison

```python
df1.union(df2)
df1.unionByName(df2)
```

## Practice

- Create two small customer queries with the same columns.
- Combine them using `UNION ALL`.
- Compare the result with `UNION`.


In [ ]:
SELECT customer_id, name, country, signup_date
FROM customers
WHERE customer_id <= 2

UNION ALL

SELECT customer_id, name, country, signup_date
FROM customers
WHERE customer_id >= 3;

# 13. CAST → Change Data Type

### Purpose

`CAST` converts a value from one data type to another.

### SQL

```sql
SELECT
    CAST(customer_id AS STRING) AS customer_id_string
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.functions import col

df = df.withColumn(
    "customer_id_string",
    col("customer_id").cast("string")
)
```

Casting is especially common when integrating data from different systems.

## Practice

- Cast `customer_id` to `STRING`.
- Cast `signup_date` to `STRING`.


In [ ]:
SELECT
    CAST(customer_id AS STRING) AS customer_id_string
FROM customers;

# 14. STRING Functions

### Purpose

SQL provides functions for cleaning and transforming text.

Common functions:

```text
UPPER()
LOWER()
TRIM()
LENGTH()
CONCAT()
SUBSTRING()
REPLACE()
```

### SQL

```sql
SELECT
    UPPER(name) AS name_upper,
    LOWER(country) AS country_lower
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.functions import upper, lower, trim

df = df.withColumn("name_upper", upper("name"))
df = df.withColumn("country_lower", lower("country"))
df = df.withColumn("name_clean", trim("name"))
```

## Practice

- Convert customer names to uppercase.
- Remove leading/trailing spaces from `name`.
- Return the length of each customer name.


In [ ]:
SELECT
    UPPER(name) AS name_upper,
    TRIM(name) AS name_clean,
    LENGTH(name) AS name_length
FROM customers;

# 15. DATE Functions

### Purpose

Date functions are used to extract parts of dates or perform date calculations.

Common functions:

```text
YEAR()
MONTH()
DAY()
DATEDIFF()
DATE_ADD()
DATE_SUB()
CURRENT_DATE()
```

### SQL

```sql
SELECT
    name,
    YEAR(signup_date) AS signup_year,
    MONTH(signup_date) AS signup_month
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.functions import year, month

df = df.withColumn("signup_year", year("signup_date"))
df = df.withColumn("signup_month", month("signup_date"))
```

## Practice

- Show each customer's signup year.
- Show each customer's signup month.
- Calculate the number of days between signup date and today.


In [ ]:
SELECT
    name,
    YEAR(signup_date) AS signup_year,
    MONTH(signup_date) AS signup_month,
    DATEDIFF(CURRENT_DATE(), signup_date) AS days_since_signup
FROM customers;

# 16. CASE WHEN → Conditional Logic

### Purpose

`CASE WHEN` is SQL's standard way to create conditional values.

### SQL

```sql
SELECT
    name,
    CASE
        WHEN country = 'UK' THEN 'Domestic'
        ELSE 'International'
    END AS customer_type
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.functions import when

df = df.withColumn(
    "customer_type",
    when(df.country == "UK", "Domestic")
    .otherwise("International")
)
```

This is one of the closest SQL ↔ PySpark comparisons.

## Practice

Create a `customer_type` column:

- UK → `Domestic`
- USA/Canada → `North America`
- Everything else → `Other`


In [ ]:
SELECT
    name,
    CASE
        WHEN country = 'UK' THEN 'Domestic'
        WHEN country IN ('USA', 'Canada') THEN 'North America'
        ELSE 'Other'
    END AS customer_type
FROM customers;

# 17. Window Functions

### Purpose

Window functions calculate values across related rows **without collapsing the rows into one row per group**.

Common functions:

```text
ROW_NUMBER()
RANK()
DENSE_RANK()
LAG()
LEAD()
SUM() OVER
AVG() OVER
```

### SQL

For example, assign a row number ordered by signup date:

```sql
SELECT
    *,
    ROW_NUMBER() OVER (
        ORDER BY signup_date
    ) AS rn
FROM customers;
```

### PySpark comparison

```python
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy("signup_date")

df = df.withColumn(
    "rn",
    row_number().over(window)
)
```

## Practice

- Add a row number ordered by `signup_date` descending.
- Create a row number partitioned by `country` and ordered by `signup_date`.


In [ ]:
SELECT
    *,
    ROW_NUMBER() OVER (
        ORDER BY signup_date DESC
    ) AS rn
FROM customers;

# 18. CTE → Common Table Expression

### Purpose

A CTE creates a temporary named result set that can be referenced by the query that follows it.

It makes complex SQL easier to read and break into logical steps.

### SQL

```sql
WITH uk_customers AS (
    SELECT *
    FROM customers
    WHERE country = 'UK'
)
SELECT *
FROM uk_customers;
```

### PySpark comparison

There is no exact one-to-one DataFrame equivalent. A common pattern is to create intermediate DataFrames:

```python
uk_customers = df.filter(df.country == "UK")
uk_customers.show()
```

## Practice

Create a CTE containing customers who signed up after `2024-02-01`, then select their names and countries.


In [ ]:
WITH recent_customers AS (
    SELECT *
    FROM customers
    WHERE signup_date > '2024-02-01'
)
SELECT
    name,
    country
FROM recent_customers;

# 19. VIEW → Reusable Query

### Purpose

A view stores a SQL query definition so it can be queried like a table.

A **temporary view** is session-scoped.

### SQL

```sql
CREATE OR REPLACE TEMP VIEW customer_summary AS
SELECT
    country,
    COUNT(*) AS customer_count
FROM customers
GROUP BY country;
```

Then:

```sql
SELECT *
FROM customer_summary;
```

### PySpark comparison

```python
df.createOrReplaceTempView("customers")
```

The PySpark DataFrame itself is not a SQL view, but this command exposes it to SQL through a temporary view.

## Practice

Create a temporary view called `uk_customers` containing only UK customers, then query it.


In [ ]:
CREATE OR REPLACE TEMP VIEW uk_customers AS
SELECT *
FROM customers
WHERE country = 'UK';

SELECT *
FROM uk_customers;

# 20. LIMIT → Restrict Number of Rows

### Purpose

`LIMIT` returns only a specified number of rows.

### SQL

```sql
SELECT *
FROM customers
LIMIT 2;
```

### PySpark comparison

```python
df.limit(2).show()
```

## Practice

- Return the first 3 customers.
- Return the latest 2 customers after sorting by `signup_date DESC`.


In [ ]:
SELECT *
FROM customers
LIMIT 2;

# SQL → PySpark Quick Comparison

| SQL | PySpark |
| --- | --- |
| `SELECT` | `select()` |
| `WHERE` | `filter()` / `where()` |
| `DISTINCT` | `distinct()` |
| `ORDER BY` | `orderBy()` |
| `GROUP BY` | `groupBy()` |
| `HAVING` | `filter()` after aggregation |
| `JOIN` | `join()` |
| `UNION ALL` | `union()` |
| `CASE WHEN` | `when().otherwise()` |
| `CAST` | `cast()` |
| `IS NULL` | `isNull()` |
| `COUNT`, `SUM`, `AVG` | `count`, `sum`, `avg` |
| Window functions | `Window` + `.over()` |
| Temporary view | `createOrReplaceTempView()` |
| `LIMIT` | `limit()` |
| `ALTER TABLE` | SQL `ALTER TABLE` / table APIs depending on operation |

### The main idea

If you understand the SQL operation, you can usually learn its PySpark equivalent by asking:

> **What DataFrame transformation produces the same result?**

The syntax changes, but the data-processing concepts remain very similar.


# Final Practice — SQL CRUD + Transformations

Using the `customers` table, complete the following without looking at the previous examples:

1. Select all customers.
2. Select only `name` and `country`.
3. Find customers from India.
4. Add a calculated `signup_year` to your query.
5. Create a `customer_type` using `CASE WHEN`.
6. Count customers by country.
7. Sort customers by signup date, newest first.
8. Create a temporary view containing customers who signed up after `2024-02-01`.
9. Use a window function to assign a row number ordered by signup date.
10. Explain which PySpark operation would be used for each of the above.

**Goal:** Focus on understanding the SQL operation first, then map it to PySpark.